## Setup and Prerequisites



## Clone the repository

```bash
git clone https://github.com/PrateekKumarSingh/presentations.git
cd ./presentations/
```

### Create a virtual environment

1. Select Python Kernel in Jupyter Notebook (top-right corner)
2. Choose "Python environment" and select "Create python environment"
3. choose "Venv Creates a 'venv' environment..."
4. Select the base interpreter (Python 3.13 or above)
5. Now select "11-How-AI-understands-meaning/requirements.txt" as the requirements file
6. Click on "OK" and wait for the environment to be created

### Activate virtual environment

```bashMacMac
.venv\Scripts\activate # Windows
source .venv/bin/activate # Linux/Mac
```

### Install SQLite viewer Extension in VSCode

https://marketplace.visualstudio.com/items?itemName=qwtel.sqlite-viewer

### Install the requirements (NOT REQUIRED if you have created the environment using Jupyter)

```bash
python3 -m pip install --upgrade pip && pip install -r requirements.txt
```


In [ ]:
import os

import azure.identity
import dotenv
import openai

# Set up OpenAI client based on environment variables
dotenv.load_dotenv()
print("Loaded environment variables from .env file")
print(dotenv.dotenv_values(".env"))
AZURE_OPENAI_SERVICE = os.getenv("AZURE_OPENAI_SERVICE")
AZURE_OPENAI_EMBEDDING_MODEL = os.getenv("AZURE_OPENAI_EMBEDDING_MODEL")
AZURE_OPENAI_TEXT_COMPLETION_MODEL = os.getenv("AZURE_OPENAI_TEXT_COMPLETION_MODEL")

azure_credential = azure.identity.AzureDeveloperCliCredential(tenant_id=os.getenv("AZURE_TENANT_ID"))
token_provider = azure.identity.get_bearer_token_provider(
    azure_credential, "https://cognitiveservices.azure.com/.default"
)
openai_client = openai.AzureOpenAI(
    api_version="2024-06-01",
    azure_endpoint=f"https://{AZURE_OPENAI_SERVICE}.openai.azure.com",
    azure_ad_token_provider=token_provider,
)

ModuleNotFoundError: No module named 'azure'

### Breaking down data in smaller chunks


In [8]:
import json
import numpy as np
from utils import format_movie  # assumes format_movie returns a string

# Load movie data
with open("/Users/prateek/workspace/repo/rag-with-azure-ai-search-notebooks/movies.json", "r") as file:
    data = json.load(file)

# Step 1: Extract movie details
movies = data["movies"]

# Step 2: Create chunks of movie details
chunk_size = 1  # Number of movies per chunk
chunks = []

for i in range(0, len(movies), chunk_size):
    chunk = movies[i : i + chunk_size]
    # format each movie into text and join with double newline if chunk_size > 1
    chunk_text = "\n\n".join(format_movie(movie) for movie in chunk)
    chunks.append(chunk_text)

# Step 3: Display the chunks
print("Chunked Movie Details:")
for i, chunk in enumerate(chunks):
    print(f"Chunk {i + 1}:\n{chunk}\n")

Chunked Movie Details:
Chunk 1:
Title: The Shawshank Redemption
Year: 1994
Genres: Crime, Drama
Plot: Two imprisoned men bond over a number of years, finding solace and eventual redemption through acts of common decency.

Chunk 2:
Title: Pulp Fiction
Year: 1994
Genres: Crime, Drama
Plot: The lives of two mob hit men, a boxer, a gangster's wife, and a pair of diner bandits intertwine in four tales of violence and redemption.

Chunk 3:
Title: Forrest Gump
Year: 1994
Genres: Comedy, Drama
Plot: Forrest Gump, while not intelligent, has accidentally been present at many historic moments, but his true love, Jenny Curran, eludes him.

Chunk 4:
Title: The Lord of the Rings: The Return of the King
Year: 2003
Genres: Action, Adventure, Drama
Plot: Gandalf and Aragorn lead the World of Men against Sauron's army to draw his gaze from Frodo and Sam as they approach Mount Doom with the One Ring.

Chunk 5:
Title: The Lord of the Rings: The Fellowship of the Ring
Year: 2001
Genres: Action, Adventure, 

### Creating Embeddings


In [9]:
chunk_embeddings = []
for chunk_text in chunks:
    response = openai_client.embeddings.create(model=AZURE_OPENAI_EMBEDDING_MODEL, input=chunk_text)
    embedding = response.data[0].embedding
    chunk_embeddings.append(embedding)

print("Chunk embeddings created:", len(chunk_embeddings), "chunks.")
print("First chunk embedding:", chunk_embeddings[0][:10])

Chunk embeddings created: 25 chunks.
First chunk embedding: [-0.03331336751580238, 0.00874294526875019, -0.02516641467809677, 0.05116203427314758, -0.010644936934113503, -0.030017271637916565, -0.00519808754324913, 0.07682597637176514, -0.009940112009644508, -0.038931239396333694]


### Why is length of all Embeddings = 1536?


In [10]:
# chunk_embeddings[1]
len(chunk_embeddings[1])

1536

# Storing the Data


### Indexing


In [11]:
import faiss

# Step 4: Index the embeddings using FAISS
dimension = len(chunk_embeddings[0])  # Dimensionality of the embeddings
index = faiss.IndexFlatL2(dimension)  # Create a flat (non-compressed) index

# Normalize embeddings first
embeddings_array = np.array(chunk_embeddings).astype("float32")
faiss.normalize_L2(embeddings_array)

# Build an IP index (inner product)
dimension = embeddings_array.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings_array)

faiss.write_index(index, "movie_title_embeddings_cosine.index")